In [1]:
# Calculate regional mortality - requires >80GB memory

In [2]:
import os
import xarray as xr
import numpy as np
import warnings
from utils.utils import get_scenario_config
from utils.mortality_utils import att_frac

In [3]:
# === Path config ===
MASKS_DIR = "/glade/work/awells/air_quality/BMR/masks/region/"
POP_DIR = "/glade/work/awells/air_quality/SSP_pop/SSP2/"
BMR_DIR = "/glade/derecho/scratch/awells/air_quality/BMR/"

In [4]:
# Load country masks
mask_file = "GBD_Region_Masks_0.10.nc"
mask_path = os.path.join(MASKS_DIR, mask_file)
masks = xr.open_dataarray(mask_path)

# Load population file
pop_file = "ssp2_coarse_grid_annual_2000-2100.nc"
pop_path = os.path.join(POP_DIR, pop_file)
population = xr.open_dataarray(pop_path)
pop = population.reindex_like(masks, method="nearest", tolerance=1e-9)

In [5]:
# === Calculate the scalar distributions (assuming normal dist.) ===
n_samples = 1000

# TMREL from GBD21 (uniform distribution)
tmrel_low = 29.1
tmrel_high = 35.7
tmrel_samples = np.random.uniform(tmrel_low, tmrel_high, size=n_samples)

tmrel_da = xr.DataArray(
    tmrel_samples,
    dims=['samples'],
    coords={'samples': np.arange(n_samples)}
).astype("float32")

# Beta from RR per 10ppb (normal distribution)
RR_10 = 1.074
RR_10_lower = 1.014
RR_10_upper = 1.137
beta_mean = np.log(RR_10) / 10
beta_std = (np.log(RR_10_upper) - np.log(RR_10_lower)) / (2 * 1.96 * 10)
beta_samples = np.random.normal(beta_mean, beta_std, size=n_samples)

beta_da = xr.DataArray(
    beta_samples,
    dims=['samples'],
    coords={'samples': np.arange(n_samples)}
).astype("float32")

# Load BMR for each grid point
bmr_file = f"GBD_BMR_Country_Mask_COPD_{n_samples}_samples_1990-2009.nc"
bmr_path = os.path.join(BMR_DIR, bmr_file)
BMR = xr.open_dataarray(bmr_path)  # three quantiles

In [6]:
warnings.filterwarnings('ignore')

# === Scenario and path config ===
# Set to whatever scenario and model you want
# Function returns error if not recognised
model = "CESM2"
scenario = "SSP245_G6"

config = get_scenario_config(model, scenario)
# ensemble_members = config["ensemble_members"][:1]
ensemble_members = [3]
years = config["years"]

O3_DIR = f"/glade/work/awells/air_quality/{model}/ozone/OSDMA8_BC/"
SAVE_DIR = f"/glade/work/awells/air_quality/{model}/mortality/ozone/region/"

for ens_num in ensemble_members:
    print(f"Processing ensemble member {ens_num:02d}")
    dates = f"{years.start}-{years.stop - 1}"

    # Load ozone data
    o3_file = f"OSDMA8_BC_{model}_{scenario}_{ens_num:02d}_{dates}.nc"
    o3_path = os.path.join(O3_DIR, o3_file)
    o3 = xr.open_dataarray(o3_path).astype("float32")
    o3 = o3.reindex_like(masks, method="nearest", tolerance=1e-9, fill_value=0)

    del o3_file, o3_path

    # make o3 dask-backed
    o3 = o3.chunk({'lat': 180, 'lon': 360})

    # range doesn't include the final year which is okay
    # because OSDMA8 excludes final year
    for year in range(2079, 2084):
        print(f"Processing year {year}")
        o3_year = o3.sel(year=year)
        AF = att_frac(o3_year, tmrel_da, beta_da).chunk({"samples": 10})

        POP = pop.sel(year=year).chunk({"lat": 180, "lon": 360})
        M = AF * BMR * POP

        del o3_year, AF, POP

        for region in masks.region:
            print(f"Processing {region.data}")

            region_mask = masks.sel(region=region)
            region_M = M.where(region_mask == 1).sum(dim=("lat", "lon"))
            del region_mask

            # mean over samples
            M_mean = region_M.mean(dim="samples")
            # median over samples
            M_median = region_M.quantile(0.5, dim="samples")
            # quantiles for box plotting
            M_25 = region_M.quantile(0.25, dim="samples")
            M_75 = region_M.quantile(0.75, dim="samples")
            # percentiles for 95% CI
            M_lower = region_M.quantile(0.025, dim="samples")
            M_upper = region_M.quantile(0.975, dim="samples")

            del region_M

            M_summary = xr.Dataset({
                "M_mean": M_mean.astype("float32").drop_vars("year"),
                "M_median": M_median.astype("float32").drop_vars("quantile"),
                "M_25": M_25.astype("float32").drop_vars("quantile"),
                "M_75": M_75.astype("float32").drop_vars("quantile"),
                "M_lower95": M_lower.astype("float32").drop_vars("quantile"),
                "M_upper95": M_upper.astype("float32").drop_vars("quantile")
            })

            del M_mean, M_median, M_25, M_75, M_lower, M_upper

            description = ("Regional mortality (COPD) due to ozone "
                           "statistics: including mean, median, "
                           "and the 95% CI - scripts by A.F. Wells (2025)")
            M_summary.attrs["description"] = description
            M_summary.attrs["model"] = model
            M_summary.attrs["scenario"] = scenario
            M_summary.attrs["ensemble_number"] = ens_num
            M_summary.attrs["region"] = region.data
            M_summary.attrs["year"] = year

            out_file = f"Regional_mortality_stats_{model}_{scenario}_{ens_num:02d}_{region.data}_{year}.nc"
            out_path = os.path.join(SAVE_DIR, out_file)
            print(f"Saving to {out_path}")
            M_summary.to_netcdf(out_path)

            del M_summary

        del M

    del o3

print("All processing complete.")

Processing ensemble member 03
Processing year 2079
Processing Central Asia
Saving to /glade/work/awells/air_quality/CESM2/mortality/ozone/region/Regional_mortality_stats_CESM2_SSP245_G6_03_Central Asia_2079.nc
Processing Central Europe
Saving to /glade/work/awells/air_quality/CESM2/mortality/ozone/region/Regional_mortality_stats_CESM2_SSP245_G6_03_Central Europe_2079.nc
Processing Eastern Europe
Saving to /glade/work/awells/air_quality/CESM2/mortality/ozone/region/Regional_mortality_stats_CESM2_SSP245_G6_03_Eastern Europe_2079.nc
Processing Australasia
Saving to /glade/work/awells/air_quality/CESM2/mortality/ozone/region/Regional_mortality_stats_CESM2_SSP245_G6_03_Australasia_2079.nc
Processing High-income Asia Pacific
Saving to /glade/work/awells/air_quality/CESM2/mortality/ozone/region/Regional_mortality_stats_CESM2_SSP245_G6_03_High-income Asia Pacific_2079.nc
Processing High-income North America
Saving to /glade/work/awells/air_quality/CESM2/mortality/ozone/region/Regional_mortalit